In [1]:
# RK4 and Blasius shooting checks. Written by Claude Code (claude-opus-5-5), 2026-10-25; see AI-USE.md § 2
# Limit: these checks verify the numerical solution of the Blasius equation, not whether Blasius
# describes the flow under a board.
import numpy as np
import pandas as pd
from shooting_blasius import (rk4_solve, blasius_rhs, F, secant_for_shooting, exact_blasius,
                              INITIAL_f, INITIAL_df, ETA_0, ETA_MAX, ETA_ARRAY, STEP_SIZE,
                              GUESS_ddf, GUESS2_ddf, TOLERANCE, white_eta, white_ddf)

pd.set_option("display.precision", 12)

In [2]:
# Limit: y'' + 4y = 0 is linear and smooth. It shows that rk4_solve is fourth order, not that RK4 suits every
# nonlinear or stiff problem; the p_obs of f''(0) in the last cell is the check on the nonlinear Blasius ODE.
# Limit: both equations are autonomous (the right-hand side ignores t and eta), so the stage times
# eta + h/2 and eta + h inside rk4_solve are never exercised: an RK4 that passes the wrong eta to k2
# still passes every check here.
# y'' + 4y = 0 as the first-order system [y, y']; y(0) = 1, y'(0) = 0 gives y = cos(2t)
T_END = 10.0
Y0 = [1.0, 0.0]


def sho_rhs(t, y):
    """Right-hand side of y'' + 4y = 0 for the state [y, dy]."""
    return np.array([y[1], -4*y[0]])


def exact_sho(t):
    """Exact solution for y(0) = 1, y'(0) = 0."""
    return np.cos(2*t)


def solve_sho(n):
    """n RK4 steps of h = T_END/n through rk4_solve. Returns the grid and y."""
    t = np.linspace(0.0, T_END, n+1)
    return t, rk4_solve(sho_rhs, Y0, t, T_END/n)[:, 0]


t, y = solve_sho(100)
print(f"h = {T_END/100}: max |y_RK4 - cos 2t| = {np.max(np.abs(y - exact_sho(t))):.3e}")

h = 0.1: max |y_RK4 - cos 2t| = 2.318e-04


In [3]:
# Limit: the table is printed to 5 decimals, so f, f' and f'' are not confirmed beyond about 1e-5
# Shoot f''(0) with the original settings, then integrate once at s*
s_star, num_iter, _, _ = secant_for_shooting(F, GUESS_ddf, GUESS2_ddf)
blasius = rk4_solve(blasius_rhs, [INITIAL_f, INITIAL_df, s_star], ETA_ARRAY, STEP_SIZE)

# Grid points that are table points (every 4th, or 2nd near eta_delta); the off-grid eta_delta row is dropped
idx = np.rint(white_eta/STEP_SIZE).astype(int)
on_grid = np.isclose(ETA_ARRAY[idx], white_eta)
table = np.array([exact_blasius(eta)[:3] for eta in ETA_ARRAY[idx[on_grid]]])
blasius_error = np.max(np.abs(blasius[idx[on_grid]] - table), axis=0)

print(f"s* = {s_star:.10f} in {num_iter} iterations; compared at {on_grid.sum()} of {len(white_eta)} table rows")
# Table rounding (half of the 5th decimal), plus 1e-6 for the RK4's own error
assert (blasius_error <= 0.5e-5*np.array([np.sqrt(2), 1.0, 1.0/np.sqrt(2)]) + 1e-6).all()
pd.DataFrame({"max |RK4 - table|": blasius_error}, index=["f", "f'", "f''"])

s* = 0.3320573508 in 7 iterations; compared at 33 of 34 table rows


,max |RK4 - table|
f,0.000006584862
f',0.000004996114
f'',0.000003439143


In [4]:
# Halve h five times; each h is solved once and reused below
N = 100*2**np.arange(6)                       # h = 0.1, 0.05, ..., 0.003125
runs = [solve_sho(n) for n in N]

# Against the exact solution: E(h)/E(h/2) = 2**p
E = np.array([np.max(np.abs(y - exact_sho(t))) for t, y in runs])
ratio = E[:-1]/E[1:]
p = np.log2(ratio)

# Oberkampf, no exact solution needed: |y_h - y_h/2| on the shared (coarse) grid points
D = np.array([np.max(np.abs(y_h - y_h2[::2])) for (_, y_h), (_, y_h2) in zip(runs, runs[1:])])
p_obs = np.log2(D[:-1]/D[1:])

print("h             =", T_END/N)
print("E(h)          =", E)
print("E(h)/E(h/2)   =", ratio)
print("p             =", p)
print("|y_h - y_h/2| =", D)
print("p_obs         =", p_obs)
assert np.allclose(ratio, 16, rtol=0.05) and np.allclose(p, 4, atol=0.1) and np.allclose(p_obs, 4, atol=0.1)

h             = [0.1      0.05     0.025    0.0125   0.00625  0.003125]
E(h)          = [2.31776636e-04 1.45952984e-05 9.32429057e-07 5.88698182e-08
 3.69727921e-09 2.31628994e-10]
E(h)/E(h/2)   = [15.8802259  15.65298542 15.83883025 15.92247026 15.96207429]
p             = [3.98915953 3.96836594 3.98539389 3.99299227 3.99657624]
|y_h - y_h/2| = [2.17524476e-04 1.36628694e-05 8.73559239e-07 5.51725390e-08
 3.46565021e-09]
p_obs         = [3.99284534 3.96721115 3.98488326 3.99275242]


In [5]:
# Limit: eta_max is tested at 10 and 20 only.
# Limit: these cases pass at h = 0.1/sqrt(2). At h = 0.2/sqrt(2) with eta_max = 20, the second guess s = 5
# makes F(5) overflow and the secant raises a ValueError, although eta_max = 10 still converges. Likely
# cause: RK4's stability limit, as the -1/2 f f'' term grows with f.
# Re-shoot with eta_max doubled, then h halved twice
cases = [(ETA_MAX, STEP_SIZE), (2*ETA_MAX, STEP_SIZE), (2*ETA_MAX, STEP_SIZE/2), (2*ETA_MAX, STEP_SIZE/4)]
rows = []
for eta_max, h in cases:
    eta = np.arange(ETA_0, eta_max, h)
    s, k, _, _ = secant_for_shooting(lambda s, eta=eta, h=h: F(s, eta, h), GUESS_ddf, GUESS2_ddf,
                                     precision=1e-12)
    rows.append((eta_max, h, s, k))

settle = pd.DataFrame(rows, columns=["eta_max", "h", "f''(0)", "secant iterations"])
settle["change from row above"] = settle["f''(0)"].diff()
settle["f''(0) - table"] = settle["f''(0)"] - white_ddf[0]

# Observed order of f''(0) itself, from the three eta_max = 20 runs
s_h, s_h2, s_h4 = settle["f''(0)"].iloc[1:]
p_obs_s = np.log2(abs(s_h - s_h2)/abs(s_h2 - s_h4))
print(f"p_obs for f''(0) = {p_obs_s:.2f}")
print(f"section 2 s* (tolerance 1e-6) minus row 0 = {s_star - settle.iloc[0, 2]:.1e}")
assert (settle["change from row above"].dropna().abs() < TOLERANCE).all() and np.isclose(p_obs_s, 4, atol=0.1)
settle

p_obs for f''(0) = 3.95
section 2 s* (tolerance 1e-6) minus row 0 = 1.1e-08


,eta_max,h,f''(0),secant iterations,change from row above,f''(0) - table
0,10.0,0.070710678119,0.332057339505,8,NaN,-0.000000004941
1,20.0,0.070710678119,0.332057338382,8,-0.000000001122,-0.000000006063
2,20.0,0.035355339059,0.332057336356,8,-0.000000002027,-0.000000008090
3,20.0,0.017677669530,0.332057336224,8,-0.000000000131,-0.000000008221
